## 2.2 信道编码与 Polar 码原理

在上一节中，我们了解了本章的学习目标和前置要求。从本节开始，我们将从信道编码的基本概念出发，理解 BPSK 调制与解调方法，然后深入 Polar 码的信道极化思想和编解码实现原理。

本节学习大纲如下：

- 信道编码为什么能改善链路性能
- BPSK 调制与解调
- Polar 码的信道极化原理
- Arikan 生成矩阵的递归构造
- SC 逐次消除解码算法

---

### 1. 信道编码的核心思想

通信系统中，发送的比特经过噪声信道后可能出错。**信道编码**通过在发送端增加冗余比特，使接收端能利用冗余信息纠正传输错误的比特。

以最简单的重复码为例：发送比特 1 时，编码器将其扩展为 "111" 发送三次。经过噪声信道后，接收端收到 "101"——虽然第二个比特被翻了，但根据"多数表决"原则（三个比特中两个是 1），接收端仍能正确恢复原始比特 1。

信道编码的本质就是以冗余换可靠性。冗余比特带来纠错能力，但也增加了传输开销——编码效率由**码率 R = 信息比特数 / 编码后总比特数**衡量。R 越小，冗余越多，纠错能力越强，但每比特传输成本越高。

---

### 2. BPSK 调制与解调

**调制：**

BPSK（Binary Phase Shift Keying）是最简单的相位调制方式，它最常用的两种相位是 0 与 $\pi$，在第 $n$ 个时隙上，其信号可以表示为：

<img src="./images/bpsk1.png" width="700">

即传输码元 1时采用与载波同相的正弦波,而传输码元 0 时采用与载波反相的正弦波。这种通过载波的不同相位来直接表示二进制数字信号的调制方式，又称为二进制绝对相移方式。其调制原理图如下所示：

<img src="./images/bpsk2.png" width="600">

本实验中 BPSK 的调制流程为：随机比特 → BPSK 星座映射 → 上采样（插零）→ RRC 脉冲成型。

PSK 调制代码如下：
```python
    def modulate(self, bits: np.ndarray) -> np.ndarray:
        """PSK调制并进行RRC脉冲成型。

        :returns: 复基带IQ信号
        """
        symbols = self.map_symbols(bits)    #比特到星座符号映射

        # 上采样
        upsampled = np.zeros(len(symbols) * self.sps, dtype=complex)
        upsampled[::self.sps] = symbols

        # RRC脉冲成型
        signal = np.convolve(upsampled, self._rrc, mode='same')
        return signal
```
其中 map_symbols() 的作用是比特到星座符号的映射。

**解调：**

得到BPSK信号以后，通过AWGN信道传递信号，再对接收到的信号进行解调和抽样，就可以完成原始信号的还原了。在这个过程中，信道中可能对信号施加的噪声、解调时选用的滤波器以及抽样的判决电平等等因素都可能会对结果产生较大影响，整个传输的示意图如下所示：

<img src="./images/bpsk3.png" width="900">

本实验中 BPSK 的解调流程为：调制信号 → AWGN 信道 → RRC 匹配滤波 → 下采样 → 判决。

PSK 解调代码如下：
```python
    def demodulate(self, signal: np.ndarray) -> np.ndarray:
        """PSK解调。

        :param signal: 复基带IQ信号

        :returns: 解调后的比特序列
        """
        # 匹配滤波
        filtered = np.convolve(signal, self._rrc, mode='same')

        # 降采样（取峰值点）
        symbols = filtered[::self.sps]

        # 反旋转 + 最近邻判决
        bps = self._bits_per_symbol
        bits = []
        for i, sym in enumerate(symbols):
            # 反相位旋转
            if self._rotation is not None and (i % 2 == 1):
                sym = sym * np.conj(self._rotation)

            # 最近邻判决
            distances = np.abs(sym - self._constellation_vals)
            idx = self._constellation_keys[np.argmin(distances)]

            # 解码比特
            bits.extend((idx >> b) & 1 for b in range(bps - 1, -1, -1))

        return np.array(bits, dtype=int)
```
在 02.03_uncoded_bpsk 章节将对调制和解调的过程进行详细分析。

---

### 3. Polar 码：信道极化

Polar 码由 Arikan 于 2009 年提出，是第一种在理论上被证明能达到香农信道容量的编码方案。其核心思想是**信道极化**——将 N 个相同的独立子信道，通过数学变换"极化"成为质量极端分化的两类：

- **K 个"极好"信道**：信道容量趋近 1，几乎可以无误传输
- **N-K 个"极差"信道**：信道容量趋近 0，几乎完全不可用

编码时只需将信息比特放在 K 个好信道上，差信道固定填入收发双方已知的 0（称为**冻结位**）。只要码长 N 足够大，好信道的比例就能逼近香农信道容量。

**信道的合成与分裂**

极化通过"合成-分裂"两步实现。首先将 N 个独立信道 $W$ 递归合成为一个大信道 $W^N$，再将这个大信道分裂回 N 个比特信道 $W_N^{(i)}$，此时各比特信道的质量已经呈两级分化。整个极化过程由 Arikan 核矩阵 $G_2$ 驱动：

$$G_2 = \begin{bmatrix} 1 & 0 \\ 1 & 1 \end{bmatrix}$$

对 N 比特的极化，生成矩阵为 $G_2$ 的 n 次 Kronecker 积（$N = 2^n$）：

$$G_N = G_2^{\otimes n}$$

编码就是将输入向量 $u = [u_0, u_1, ..., u_{N-1}]$（包含信息位和冻结位）乘以生成矩阵：

$$x = u \cdot G_N$$

其中乘法在 GF(2) 上执行。蝶形编码的复杂度仅为 $O(N\log N)$，远低于通用线性分组码的 $O(N^2)$。

**可靠性序列**

实际编码中如何确定哪些位置是好信道？3GPP 标准通过密度进化等数值方法预先计算了各码长下的**可靠性序列**——将 N 个子信道按错误概率从小到大排列。编码时取序列末尾的 K 个作为信息位，其余为冻结位。

#### 3.1 PolarEncoder 实现

`PolarEncoder` 在初始化时完成三件事：从可靠性序列中取末尾 K 个作为信息位、其余 N-K 个作为冻结位、预分配编码所需数组。

```python
class PolarEncoder:
    def __init__(self, N: int, K: int) -> None:
        self.N = N
        self.K = K
        self.frozen_set, self.info_set = _get_frozen_and_info_sets(N, K)
        seq = _get_reliability_sequence(N)          # 预计算的可靠性序列
        self.info_positions = sorted(seq[N - K :])   # 末尾 K 个 = 好信道
        self._info_pos_arr = np.array(self.info_positions, dtype=np.intp)
```

其中 `_get_frozen_and_info_sets(N, K)` 根据可靠性序列返回冻结集（frozen_set）和信息集（info_set），编码时据此判断每个位置的角色。

**encode(info_bits)** — 编码入口。将 K 个信息比特放置到好信道位置（冻结位置填 0），然后通过蝶形 GF(2) 变换生成 N 位码字：

```python
def encode(self, info_bits: np.ndarray) -> np.ndarray:
    # 构建编码输入 u: 信息位填到好信道, 其余位置为 0 (冻结)
    u = np.zeros(self.N, dtype=np.int8)
    u[self._info_pos_arr] = info_bits             # 向量化插入
    # 蝶形 GF(2) 编码: x = u * G_N
    d = u.copy()
    stage = 1
    while stage < self.N:                           # 逐级蝶形变换
        for j in range(0, self.N, 2 * stage):
            d[j:j + stage] ^= d[j + stage:j + 2 * stage]  # GF(2) 异或
        stage <<= 1                                  # 步长翻倍
    return d
```

蝶形变换等价于乘以 Arikan 核矩阵 G2 的 Kronecker 积。N=8 时三级蝶形（stage=1,2,4）完成 x = u * G8，复杂度 O(N log N)。

实际使用：

```python
enc = PolarEncoder(N=256, K=112)   # 码长 256, 信息位 112 (码率约 1/2)
coded = enc.encode(info_bits)       # K bits -> N bits
```

---

### 4. SC 逐次消除解码

Polar 码的标准解码算法是 **SC（Successive Cancellation，逐次消除）解码**，工作在 LLR 域，按 $u_0, u_1, ..., u_{N-1}$ 的顺序逐比特判决。

**解码规则**

- 若位置 $i$ 为冻结位：直接判 $\hat{u}_i = 0$（收发双方预先已知）
- 若位置 $i$ 为信息位：根据当前 LLR 和已解码的前 $i-1$ 个比特进行似然比判决

这种"前序判决作为后序已知条件"的机制就是"逐次消除"的含义——前面的比特一旦判完，后面的比特就能利用这些已知信息来提高判决精度。

**蝶形递推**

SC 解码通过蝶形结构递归计算各级的 LLR 和部分和，核心是两个基本操作：

**f 操作**（从右向左，组合两个子节点的 LLR）：

$$f(L_a, L_b) = \text{sign}(L_a) \cdot \text{sign}(L_b) \cdot \min(|L_a|, |L_b|)$$

f 操作的物理含义：当两个分支的 LLR 符号相同时，整体 LLR 取同号且幅度为较小者（因为两路一致时置信度降低）；符号不同时取负号（倾向于 1）。

**g 操作**（从右向左，结合左侧已解码比特 $\hat{u}_\text{left}$ 更新右侧 LLR）：

$$g(L_a, L_b, \hat{u}_\text{left}) = L_b + (1 - 2\hat{u}_\text{left}) \cdot L_a$$

g 操作的物理含义：已知左侧比特 $\hat{u}_\text{left}$ 后，可以对右侧 LLR 进行修正——若左侧判为 0，右侧 LLR 加上左 LLR（增强置信度）；若左侧判为 1，右侧 LLR 减去左 LLR。

**解码流程示例**

以 N=8 为例，解码树从根节点（N=8 的 LLR 层）出发：

1. 通过 f 操作逐层向左下行直到叶节点，得到当前位置的 LLR
2. 对当前位置做出判决（冻结位判 0，信息位根据 LLR 符号判 0/1）
3. 将判决结果通过 g 操作向右上行，更新父节点的部分和
4. 用更新后的部分和继续处理下一个比特

#### PolarDecoder 实现

`PolarDecoder` 在初始化时预分配两个二维数组 —— `_L`（LLR，每层 N 个）和 `_B`（部分和，每层 N 个），避免递归中频繁分配内存。`_is_frozen` 标记每个位置是否为冻结位。

```python
class PolarDecoder:
    def __init__(self, N: int, K: int) -> None:
        self.N = N; self.K = K
        self.n = int(np.log2(N))              # 蝶形层数, N=256 -> n=8
        self.frozen_set, self.info_set = _get_frozen_and_info_sets(N, K)
        seq = _get_reliability_sequence(N)
        self.info_positions = sorted(seq[N - K :])
        self._is_frozen = np.ones(N, dtype=bool)
        for pos in self.info_positions:
            self._is_frozen[pos] = False      # 信息位标记为 False
        # 预分配 (n+1) 层 LLR 和部分和数组
        self._L = np.zeros((self.n + 1, N), dtype=np.float64)
        self._B = np.zeros((self.n + 1, N), dtype=np.int8)
```

**decode(llr)** — SC 解码入口。接收 N 个信道 LLR，逐比特判决输出 N 个比特。内部用迭代（非递归）实现蝶形 f/g 递推，逐层向上/向下更新 `_L` 和 `_B`。

**get_polar_decoder(N, K)** — 工厂函数，返回配置好的 PolarDecoder 实例：

```python
def get_polar_decoder(n: int, k: int) -> PolarDecoder:
    return PolarDecoder(n, k)
```

实际使用：

```python
dec = get_polar_decoder(256, 112)    # 创建 SC 译码器
decoded = dec.decode(llr)             # N 个 LLR -> N 个比特
```

**可靠性序列说明**

`_get_reliability_sequence(N)` 返回特定码长下预计算的子信道可靠性排序。以 N=256、K=112 为例，选取序列末尾 112 个位置作为信息位 —— 这些位置的子信道经过极化后质量最好。码率 = K/N = 112/256 ≈ 1/2。

---
### 练习

1. Polar 码的核心思想是什么？
   - A. 通过增加发射功率提升信噪比
   - B. 通过信道极化将子信道分化为极好和极差两类，信息位放在好信道上
   - C. 通过多次重传同一帧来提高可靠性
   - D. 通过增大调制阶数来提高频谱效率

2. Polar 编码中，冻结位的作用是什么？
   - A. 存储 CRC 校验信息
   - B. 填入收发双方已知的固定值（通常为 0），占据极化后的差信道
   - C. 存储同步序列
   - D. 作为导频符号插入帧中

3. SC 解码中，f 和 g 操作分别完成什么功能？
   - A. f 为编码，g 为译码
   - B. f 从右向左计算 LLR，g 结合已解码比特修正后续 LLR
   - C. f 计算 CRC，g 进行 Polar 编码
   - D. f 和 g 均为信道估计操作

4. 低 SNR 下 Polar 编码的 BER 反高于无编码 BPSK，原因是什么？
   - A. 码率补偿使每信息比特 SNR 降低，且 SC 解码错误会传播
   - B. Polar 编码本身引入了额外的比特错误
   - C. 无编码 BPSK 的调制方式更优
   - D. AWGN 信道对编码信号的影响更大


执行以下代码获取答案


In [ ]:
!cat answer/02.02_answer.txt
